In [2]:
## imports
import pandas as pd
import numpy as np
import re
import requests
import yaml


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 1. Example 1: no credentials; no wrapper

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

## 1.1 Query to pull some data

In [3]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

In [4]:
## use requests to call the api
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))

## get the json contents of the response 
## here, we're assuming valid response
naep_resp_j = naep_resp.json()
naep_resp_j

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

<Response [200]>

<class 'requests.models.Response'>


{'status': 200,
 'serviceVersion': '6.4.2026.1',
 'dwellTimeMS': '0',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_DATABASE',
 'Source': 'B11B',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDisplayab

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


## 1.2 What happens if there's an error in our query?

In [5]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [6]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

<Response [400]>

In [8]:
## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
# naep_wrong_resp.json() # uncomment to see error

### 1.2.2 More all-purpose way of allowing remainder of calls to run: try, except

In [9]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except Exception as e:
    print('Failed to get result from API due to error:')
    print(e) # or just: pass

Failed to get result from API due to error:
Invalid control character at: line 1 column 293 (char 292)


### 1.2.3 Can usually also find more targeted way but that varies more across APIs

In [10]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along those lines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2733\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2348\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 947\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 354"}'

'{"status":200,"serviceVersion":"6.4.2026.1","dwellTimeMS":"0","avgWebHostCPUTotalLoad":"N/A","dataHitType":"FROM_DATABASE","Source":"B11B","result": [{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"1","varValueLabel":"Male","value":139.099504632971,"isStatDisplayable":1,"errorFlag":0},{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"2","varValueLabel":"Female","value":158.567104984955,"isStatDisplayable":1,"errorFlag":0}]}'

NAEP results not found


## Activity 1: writing a function to make multiple, sequential calls

- Say we want to pull the data for grades 4, 8, and 12
- How can we write a function that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [14]:
# your code here
def make_call(grade):
    query = (
        "https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?"
        "type=data&subject=writing&grade={}&"
        "subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011"
    ).format(grade)

    naep_resp = requests.get(query)
    
    ## get the json contents of the response 
    ## here, we're assuming valid response
    naep_resp_j = naep_resp.json()
    
    ## with result, turn it into a dataframe
    return pd.DataFrame(naep_resp_j['result'])
grades = [4, 8, 12]
grade_data = pd.concat([make_call(g) for g in grades])
    

JSONDecodeError: Invalid control character at: line 1 column 293 (char 292)

In [16]:
# Define grades to loop through
grades = [4, 8, 12]


def make_call(grade):
    # Construct the API query URL
    url = (
        "https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?"
        "type=data&subject=writing&grade={}&"
        "subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011"
    ).format(grade)

    try:
        response = requests.get(url)
        response.raise_for_status()

        # Handle potential control character issues by disabling strict mode
        data = json.loads(response.text, strict=False)

        # The NAEP API returns a dictionary with status and a 'result' list
        if "result" in data:
            df = pd.DataFrame(data["result"])
            # Add a column to keep track of which grade this row represents
            df["requested_grade"] = grade
            return df

    except Exception as e:
        print(f"Error fetching data for grade {grade}: {e}")

    return pd.DataFrame()  # Return empty DataFrame on failure


# Combine data from all grades into a single DataFrame
grade_data = pd.concat([make_call(g) for g in grades], ignore_index=True)

# Display the resulting DataFrame structure
grade_data

Error fetching data for grade 4: 400 Client Error: Bad Request for url: https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=4&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011


,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag,requested_grade
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0,8
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0,8
2,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,1,Male,141.256978,1,0,12
3,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,2,Female,155.385917,1,0,12


# 2. Example 2: needs credentials; no wrapper

Create an account here: https://www.yelp.com/developers/v3/manage_app

In [36]:
## get the key
API_KEY = ""

In [19]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Hanover,NH,03755"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [20]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf.head()

{'id': '8ybF6YyRldtZmU9jil4xlg',
 'alias': 'mollys-restaurant-and-bar-hanover',
 'name': "Molly's Restaurant & Bar",
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA6z-SnPgZfrs2GQNQ/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/mollys-restaurant-and-bar-hanover?adjust_creative=lJDCWlxRHFvsysUSczxudA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=lJDCWlxRHFvsysUSczxudA',
 'review_count': 582,
 'categories': [{'alias': 'tradamerican', 'title': 'American'},
  {'alias': 'burgers', 'title': 'Burgers'},
  {'alias': 'pizza', 'title': 'Pizza'}],
 'rating': 3.9,
 'coordinates': {'latitude': 43.701144, 'longitude': -72.2894249},
 'transactions': ['delivery'],
 'price': '$$',
 'location': {'address1': '43 South Main St',
  'address2': '',
  'address3': '',
  'city': 'Hanover',
  'zip_code': '03755',
  'country': 'US',
  'state': 'NH',
  'display_address': ['43 South Main St', 'Hanover, NH 03755']},
 'phone': '+16036432570',
 'display_phone': '(

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,price,location,phone,display_phone,distance,business_hours,attributes
0,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,False,https://www.yelp.com/biz/mollys-restaurant-and...,582,"[{'alias': 'tradamerican', 'title': 'American'...",3.9,"{'latitude': 43.701144, 'longitude': -72.2894249}",[delivery],$$,"{'address1': '43 South Main St', 'address2': '...",+16036432570,(603) 643-2570,250.830160,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://www.mollysrestaurant.com/...
1,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,False,https://www.yelp.com/biz/little-havana-hanover...,29,"[{'alias': 'cuban', 'title': 'Cuban'}, {'alias...",4.9,"{'latitude': 43.700743, 'longitude': -72.287599}","[delivery, pickup]",NaN,"{'address1': '15 Lebanon St', 'address2': '', ...",+18383831000,(838) 383-1000,102.833229,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.canva.com/design/DAG...
2,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/tScZeo...,False,https://www.yelp.com/biz/base-camp-cafe-hanove...,264,"[{'alias': 'himalayan', 'title': 'Himalayan/Ne...",4.4,"{'latitude': 43.700626, 'longitude': -72.2887803}",[delivery],$$,"{'address1': '3 Lebanon St', 'address2': 'Ste ...",+16036432007,(603) 643-2007,196.139758,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://basecampcafenewhampshire....
3,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,False,https://www.yelp.com/biz/bistro-at-six-hanover...,2,"[{'alias': 'lounges', 'title': 'Lounges'}, {'a...",4.0,"{'latitude': 43.7001146, 'longitude': -72.2877...",[],$$,"{'address1': '6 E South St', 'address2': '', '...",+16036430600,(603) 643-0600,198.651788,"[{'open': [{'is_overnight': True, 'start': '00...",{}
4,1Q9gTry0GH2NFA7O398xeA,casa-brava-tapas-hanover,Casa Brava Tapas,https://s3-media0.fl.yelpcdn.com/bphoto/fA3__V...,False,https://www.yelp.com/biz/casa-brava-tapas-hano...,10,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.8,"{'latitude': 43.70019133305323, 'longitude': -...",[],NaN,"{'address1': '6 South St', 'address2': '', 'ad...",+16038509763,(603) 850-9763,202.701925,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://casabravatapas.com/tapas...


In [21]:
## more data-specific way of summarizing
## we're doing a simple approach and just retaining
## cols that have a simple str structure
## if doing for real, would want to extract things
def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)


,id,alias,name,image_url,url,price,phone,display_phone
8ybF6YyRldtZmU9jil4xlg,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,https://www.yelp.com/biz/mollys-restaurant-and...,$$,+16036432570,(603) 643-2570
JFE0XffhpP3Bi3DQRvymVg,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,https://www.yelp.com/biz/little-havana-hanover...,NaN,+18383831000,(838) 383-1000
XVGEEIH5rVB2QzW-qywcJw,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/tScZeo...,https://www.yelp.com/biz/base-camp-cafe-hanove...,$$,+16036432007,(603) 643-2007
wyV_NfYn4ZOfp_sHMDPcAw,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,https://www.yelp.com/biz/bistro-at-six-hanover...,$$,+16036430600,(603) 643-0600
1Q9gTry0GH2NFA7O398xeA,1Q9gTry0GH2NFA7O398xeA,casa-brava-tapas-hanover,Casa Brava Tapas,https://s3-media0.fl.yelpcdn.com/bphoto/fA3__V...,https://www.yelp.com/biz/casa-brava-tapas-hano...,NaN,+16038509763,(603) 850-9763
KA8yhrd-ClVYMyOefXdVYg,KA8yhrd-ClVYMyOefXdVYg,lous-restaurant-and-bakery-hanover,Lou's Restaurant & Bakery,https://s3-media0.fl.yelpcdn.com/bphoto/ZguQWu...,https://www.yelp.com/biz/lous-restaurant-and-b...,$$,+16036433321,(603) 643-3321
5WW4g_LRwau29KyjZGLyAA,5WW4g_LRwau29KyjZGLyAA,sawtooth-kitchen-hanover,Sawtooth Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/61MNG4...,https://www.yelp.com/biz/sawtooth-kitchen-hano...,NaN,+16036435134,(603) 643-5134


# Activity 2: pull restaurants in a different location

- Try running a business search query for your hometown or another place by constructing a query similar to `yelp_genquery` but changing the location parameter
- Other endpoints require feeding what's called the business' fusion id into the API. Take an id from `yelp_stronly.id` and use the documentation here to pull the reviews for that business: https://docs.developer.yelp.com/reference/v3_business_reviews
- **Challenge**: generalize the previous step by writing a function that (1) takes a list of business ids as an input, (2) calls the reviews API for each id, (3) returns the results, and (4) rowbinds all results, i.e. turns them into a single, usable DataFrame

In [41]:
# your code here
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "New York,NY,10022"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [42]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf.head()

{'id': 'gZz9A8k8ORC_xl0aHxtY4w',
 'alias': 'monkey-bar-new-york-4',
 'name': 'Monkey Bar',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/bBVe73440lsk-HErXjHOkw/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/monkey-bar-new-york-4?adjust_creative=lJDCWlxRHFvsysUSczxudA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=lJDCWlxRHFvsysUSczxudA',
 'review_count': 506,
 'categories': [{'alias': 'tradamerican', 'title': 'American'},
  {'alias': 'bars', 'title': 'Bars'}],
 'rating': 4.3,
 'coordinates': {'latitude': 40.75998, 'longitude': -73.97313},
 'transactions': [],
 'location': {'address1': '60 E 54th St',
  'address2': '',
  'address3': '',
  'city': 'New York',
  'zip_code': '10022',
  'country': 'US',
  'state': 'NY',
  'display_address': ['60 E 54th St', 'New York, NY 10022']},
 'phone': '+12124040365',
 'display_phone': '(212) 404-0365',
 'distance': 456.93567712264223,
 'business_hours': [{'open': [{'is_overnight': False,
     'start': '1

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,location,phone,display_phone,distance,business_hours,attributes,price
0,gZz9A8k8ORC_xl0aHxtY4w,monkey-bar-new-york-4,Monkey Bar,https://s3-media0.fl.yelpcdn.com/bphoto/bBVe73...,False,https://www.yelp.com/biz/monkey-bar-new-york-4...,506,"[{'alias': 'tradamerican', 'title': 'American'...",4.3,"{'latitude': 40.75998, 'longitude': -73.97313}",[],"{'address1': '60 E 54th St', 'address2': '', '...",+12124040365,(212) 404-0365,456.935677,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.nycmonkeybar.com/men...,NaN
1,VmdyRRMtUXOWV7F7w0ImwQ,piccola-cucina-uptown-new-york,Piccola Cucina Uptown,https://s3-media0.fl.yelpcdn.com/bphoto/McMmjH...,False,https://www.yelp.com/biz/piccola-cucina-uptown...,628,"[{'alias': 'sicilian', 'title': 'Sicilian'}]",4.4,"{'latitude': 40.763174, 'longitude': -73.969046}","[delivery, pickup]","{'address1': '106 E 60th St', 'address2': '', ...",+16467073997,(646) 707-3997,545.563687,"[{'open': [{'is_overnight': False, 'start': '1...",{},$$$
2,CIzl1SAhtNUGEkLgvTB9zQ,rosemarys-midtown-new-york-2,Rosemary's - Midtown,https://s3-media0.fl.yelpcdn.com/bphoto/9vKP11...,False,https://www.yelp.com/biz/rosemarys-midtown-new...,178,"[{'alias': 'italian', 'title': 'Italian'}, {'a...",4.1,"{'latitude': 40.75588334611095, 'longitude': -...","[delivery, pickup]","{'address1': '825 Third Ave', 'address2': '', ...",+12128706137,(212) 870-6137,315.799006,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.rosemarysnyc.com/mid...,$$
3,C-8mGN7lt5rIraO5n-15ug,the-smith-new-york-2,The Smith,https://s3-media0.fl.yelpcdn.com/bphoto/jPEpAf...,False,https://www.yelp.com/biz/the-smith-new-york-2?...,3346,"[{'alias': 'newamerican', 'title': 'New Americ...",4.0,"{'latitude': 40.75520199, 'longitude': -73.967...","[delivery, pickup]","{'address1': '956 2nd Ave', 'address2': '', 'a...",+12126442700,(212) 644-2700,346.016294,"[{'open': [{'is_overnight': False, 'start': '0...",{'menu_url': 'https://thesmithrestaurant.com/l...,$$$
4,krZunArZFdp_2G1arwVuVw,au-zaatar-midtown-east-new-york-5,Au Za'atar - Midtown East,https://s3-media0.fl.yelpcdn.com/bphoto/zbMyWl...,False,https://www.yelp.com/biz/au-zaatar-midtown-eas...,590,"[{'alias': 'lebanese', 'title': 'Lebanese'}]",4.5,"{'latitude': 40.759, 'longitude': -73.962771}","[delivery, pickup]","{'address1': '1063 1st Ave', 'address2': '', '...",+12126253982,(212) 625-3982,466.891032,"[{'open': [{'is_overnight': True, 'start': '10...",{},NaN


In [40]:
business = 'be1z3HdjBqntCpsk8ThpCQ'
url = ("https://api.yelp.com/v3/businesses/{}/reviews?limit=20&sort_by=yelp_sort").format(business)

headers = {'Authorization': f"Bearer {API_KEY}"}

yelp_genresp = requests.get(url, headers = headers)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()
print(yelp_genjson)
#yelp_gendf = pd.DataFrame(yelp_genjson['reviews'])
#yelp_gendf.head()

<Response [404]>

{'error': {'code': 'NOT_FOUND', 'description': 'Resource could not be found.'}}
